# Agentic RAG

An agent loop that decides when to search, what to search for and when it has enough evidence.

**Extra requirements:** Optional: a Contextual AI account if you want to swap in their hosted agent platform

Uses the shared helpers in `../common.py` (local embeddings + Ollama). Make sure `ollama serve` is running and `ollama pull llama3.2` is done.

In [ ]:
import sys
sys.path.insert(0, "..")  # makes common.py importable
from common import *

text = load_text()
chunks = chunk_text(text, size=500, overlap=50)
store = build_store(chunks)
print(f"{len(chunks)} chunks indexed")

Instead of one fixed retrieve-then-generate pass, the LLM runs a **reason -> act -> observe** loop with tools. This notebook uses a local agent; the same loop can be backed by a hosted platform such as Contextual AI (needs an account and API key).

In [ ]:
def search_tool(q):
    return "\n".join(f"- {h['text']}" for h in store.search(q, 3))

def calculator_tool(expr):
    if not re.fullmatch(r"[0-9+\-*/(). ]+", expr):
        return "invalid expression"
    return str(eval(expr, {"__builtins__": {}}))

TOOLS = {"search": search_tool, "calculator": calculator_tool}

AGENT_PROMPT = """You answer questions using tools.
Tools:
- search: look up the handbook. Input: a search query.
- calculator: do arithmetic. Input: an expression.
Reply with ONE JSON object each turn:
{"thought": "...", "action": "search" | "calculator" | "final", "input": "..."}
Use "final" with the answer text as input once you have enough evidence."""

In [ ]:
def agent(question, max_steps=5, verbose=True):
    scratch = f"Question: {question}\n"
    for step in range(max_steps):
        move = llm_json(AGENT_PROMPT + "\n\n" + scratch)
        if not isinstance(move, dict) or "action" not in move:
            break
        if verbose: print(f"step {step + 1}: {move.get('thought', '')}\n   -> {move['action']}({move.get('input', '')!r})")
        if move["action"] == "final":
            return move["input"]
        tool = TOOLS.get(move["action"])
        observation = tool(move["input"]) if tool else "unknown tool"
        scratch += f"Action: {move['action']}({move['input']})\nObservation: {observation}\n"
    return llm(scratch + "\nGive the best final answer from the observations above.")

print(agent("How many dimensions does all-MiniLM-L6-v2 produce, and how many floats is that for 1000 documents?"))

**Trade-off:** more flexible and able to do multi-hop research, but costs more tokens, adds latency and needs step limits plus logging to stay debuggable.